# 11 · Retrieval Methods: BM25, Hybrid, SPLADE, ColBERT

Sparse, hybrid, learned-sparse and late-interaction retrieval, side by side.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfzalManihar/Agentic-AI-Learning/blob/main/11_retrieval_methods.ipynb)

**Covered in this notebook**

- Retrieval
- Sparse Retrieval
- Hybrid search retrieval

> Part of [Agentic AI Learning](https://github.com/AfzalManihar/Agentic-AI-Learning). Add your own API keys in Colab Secrets (key icon) before running.

#27-09-2006

# Retrieval

## You have already done Dense Retrieval


# Sparse Retrieval

In [ ]:
!pip install -U rank-bm25 -q


text = """
Photosynthesis is the process plants use to convert sunlight into glucose.
It requires carbon dioxide, water, and light energy.
Chlorophyll in the chloroplasts absorbs the light.

The French Revolution began in 1789 and fundamentally changed European society.
Louis XVI was executed in 1793. The revolution introduced ideals of liberty and equality.
Napoleon rose to power in its aftermath and reshaped the continent.

Machine learning is a subset of artificial intelligence.
Neural networks learn patterns from data through gradient descent.
Transformers have become the dominant architecture since 2017.
"""

In [ ]:
documents = text.strip().split("\n\n")

for i, doc in enumerate(documents):
    print(f"\nDocument {i + 1}:")
    print(doc)


Document 1:
Photosynthesis is the process plants use to convert sunlight into glucose.
It requires carbon dioxide, water, and light energy.
Chlorophyll in the chloroplasts absorbs the light.

Document 2:
The French Revolution began in 1789 and fundamentally changed European society.
Louis XVI was executed in 1793. The revolution introduced ideals of liberty and equality.
Napoleon rose to power in its aftermath and reshaped the continent.

Document 3:
Machine learning is a subset of artificial intelligence.
Neural networks learn patterns from data through gradient descent.
Transformers have become the dominant architecture since 2017.


In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize documents
tokenized_documents = [
    doc.lower().split()
    for doc in documents
]

# Create BM25 retriever
bm25 = BM25Okapi(tokenized_documents)

In [ ]:
query = "What happened during the French Revolution?"

# Tokenize query
tokenized_query = query.lower().split()

# Retrieve relevant documents
results = bm25.get_top_n(
    tokenized_query,
    documents,
    n=2
)

for i, result in enumerate(results):
    print(f"\n--- Result {i + 1} ---")
    print(result)


--- Result 1 ---
The French Revolution began in 1789 and fundamentally changed European society.
Louis XVI was executed in 1793. The revolution introduced ideals of liberty and equality.
Napoleon rose to power in its aftermath and reshaped the continent.

--- Result 2 ---
Photosynthesis is the process plants use to convert sunlight into glucose.
It requires carbon dioxide, water, and light energy.
Chlorophyll in the chloroplasts absorbs the light.


In [ ]:
scores = bm25.get_scores(tokenized_query)

for i, score in enumerate(scores):
    print(f"Document {i + 1}: {score:.4f}")

Document 1: 0.1729
Document 2: 0.6196
Document 3: 0.1078


# Hybrid search retrieval


In [ ]:
!pip install -q rank-bm25 sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 91.3 MB/s eta 0:00:00


In [ ]:
text = """
Python is widely used for data science and machine learning.
Pandas is a Python library used for data analysis.
SQL is used to query and manage databases.
Machine learning models learn patterns from data.
Photosynthesis converts sunlight into chemical energy.
The French Revolution began in 1789.
"""

documents = text.strip().split("\n")

query = "How is Python used in machine learning?"

In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize documents
tokenized_documents = [
    doc.lower().split()
    for doc in documents
]

# Create BM25 retriever
bm25 = BM25Okapi(tokenized_documents)

# Tokenize query
tokenized_query = query.lower().split()

# Calculate BM25 scores
sparse_scores = bm25.get_scores(tokenized_query)

for i, score in enumerate(sparse_scores):
    print(f"Document {i + 1}: {score:.4f}")

Document 1: 1.0340
Document 2: 0.5451
Document 3: 0.0000
Document 4: 0.6117
Document 5: 0.0000
Document 6: 1.4402


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Create document embeddings
document_embeddings = model.encode(documents)

# Create query embedding
query_embedding = model.encode([query])

# Calculate cosine similarity
similarities = np.dot(
    document_embeddings,
    query_embedding[0]
) / (
    np.linalg.norm(document_embeddings, axis=1)
    * np.linalg.norm(query_embedding[0])
)

for i, score in enumerate(similarities):
    print(f"Document {i + 1}: {score:.4f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Document 1: 0.8101
Document 2: 0.5139
Document 3: 0.2948
Document 4: 0.5087
Document 5: 0.0804
Document 6: 0.0462


In [ ]:
# Normalize BM25 scores
sparse_scores = (
    sparse_scores - sparse_scores.min()
) / (
    sparse_scores.max() - sparse_scores.min() + 1e-8
)

# Combine Sparse + Dense scores
alpha = 0.5

hybrid_scores = (
    alpha * sparse_scores
    + (1 - alpha) * similarities
)

# Rank documents
results = sorted(
    zip(documents, hybrid_scores),
    key=lambda x: x[1],
    reverse=True
)

for i, (doc, score) in enumerate(results, 1):
    print(f"\n--- Rank {i} | Score: {score:.4f} ---")
    print(doc)


--- Rank 1 | Score: 0.7640 ---
Python is widely used for data science and machine learning.

--- Rank 2 | Score: 0.5231 ---
The French Revolution began in 1789.

--- Rank 3 | Score: 0.4667 ---
Machine learning models learn patterns from data.

--- Rank 4 | Score: 0.4462 ---
Pandas is a Python library used for data analysis.

--- Rank 5 | Score: 0.1474 ---
SQL is used to query and manage databases.

--- Rank 6 | Score: 0.0402 ---
Photosynthesis converts sunlight into chemical energy.


#SPLADE search retrieval

In [ ]:
!pip install -q transformers torch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

MODEL_NAME = "naver/splade-cocondenser-ensembledistil"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

model.eval()

print("SPLADE model loaded successfully!")

config.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

SPLADE model loaded successfully!


In [ ]:
def splade_vector(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    )

    with torch.no_grad():
        output = model(**inputs).logits

    # SPLADE sparse representation
    sparse_vector = torch.max(
        torch.log1p(torch.relu(output)),
        dim=1
    ).values

    return sparse_vector[0]

print("SPLADE vector function ready!")

SPLADE vector function ready!


In [ ]:
documents = [
    "I bought a new car yesterday.",
    "The vehicle has a powerful engine.",
    "Python is used for machine learning.",
    "The automobile requires regular maintenance."
]

query = "car maintenance"

print("Documents:", len(documents))
print("Query:", query)

Documents: 4
Query: car maintenance


In [ ]:
document_vectors = [
    splade_vector(doc)
    for doc in documents
]

print("Document vectors created:", len(document_vectors))
print("Vector dimension:", document_vectors[0].shape)

Document vectors created: 4
Vector dimension: torch.Size([30522])


In [ ]:
query_vector = splade_vector(query)

print("Query vector created!")
print("Vector dimension:", query_vector.shape)

Query vector created!
Vector dimension: torch.Size([30522])


In [ ]:
scores = []

for i, doc_vector in enumerate(document_vectors):

    score = torch.dot(
        query_vector,
        doc_vector
    ).item()

    scores.append((documents[i], score))

print("Similarity scores calculated!")

Similarity scores calculated!


In [ ]:
results = sorted(
    scores,
    key=lambda x: x[1],
    reverse=True
)

for i, (doc, score) in enumerate(results, 1):

    print(f"\n--- Rank {i} | Score: {score:.4f} ---")
    print(doc)


--- Rank 1 | Score: 23.5397 ---
The automobile requires regular maintenance.

--- Rank 2 | Score: 11.2269 ---
I bought a new car yesterday.

--- Rank 3 | Score: 7.7974 ---
The vehicle has a powerful engine.

--- Rank 4 | Score: 0.1570 ---
Python is used for machine learning.


#COLBERT

In [ ]:
!pip install -q transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "colbert-ir/colbertv2.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name);

model.eval()

print("COLBERT model loaded successfully!")

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key           | Status     |  | 
--------------+------------+--+-
linear.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


COLBERT model loaded successfully!


In [ ]:
documents = [
    "Python is widely used for machine learning.",
    "Pandas is used for data analysis.",
    "SQL is used to manage databases.",
    "Machine learning models learn patterns from data."
]

query = "Python machine learning"

In [ ]:
def get_embeddings(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state

    return embeddings

In [ ]:
query_embedding = get_embeddings(query)

scores = []

for document in documents:

    document_embedding = get_embeddings(document)

    # Token-to-token similarity
    similarity = torch.matmul(
        query_embedding,
        document_embedding.transpose(1, 2)
    )

    # MaxSim: best matching document token for each query token
    max_similarity = similarity.max(dim=2).values

    # Final COLBERT score
    score = max_similarity.sum().item()

    scores.append((document, score))

In [ ]:
results = sorted(
    scores,
    key=lambda x: x[1],
    reverse=True
)

for i, (document, score) in enumerate(results, 1):
    print(f"\n--- Rank {i} | Score: {score:.4f} ---")
    print(document)


--- Rank 1 | Score: 671.1689 ---
Python is widely used for machine learning.

--- Rank 2 | Score: 565.3956 ---
Machine learning models learn patterns from data.

--- Rank 3 | Score: 335.3419 ---
SQL is used to manage databases.

--- Rank 4 | Score: 315.7159 ---
Pandas is used for data analysis.


#Ensemble Retrieval = BM25 + Dense Retrieval

In [ ]:
!pip install -q rank-bm25 sentence-transformers

In [ ]:
documents = [
    "Python is widely used for machine learning.",
    "Pandas is used for data analysis.",
    "SQL is used to manage databases.",
    "Machine learning models learn patterns from data.",
    "Python can be used to build AI applications."
]

query = "How is Python used in machine learning?"

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_documents = [
    doc.lower().split()
    for doc in documents
]

bm25 = BM25Okapi(tokenized_documents)

tokenized_query = query.lower().split()

sparse_scores = bm25.get_scores(tokenized_query)

for i, score in enumerate(sparse_scores):
    print(f"Document {i + 1}: {score:.4f}")

Document 1: 1.0744
Document 2: 0.4389
Document 3: 0.4389
Document 4: 0.3321
Document 5: 0.5043


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

document_embeddings = model.encode(documents)
query_embedding = model.encode(query)

dense_scores = np.dot(
    document_embeddings,
    query_embedding
) / (
    np.linalg.norm(document_embeddings, axis=1)
    * np.linalg.norm(query_embedding)
)

for i, score in enumerate(dense_scores):
    print(f"Document {i + 1}: {score:.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Document 1: 0.8676
Document 2: 0.4422
Document 3: 0.2928
Document 4: 0.5087
Document 5: 0.5996


In [ ]:
# Normalize BM25 scores
sparse_scores = (
    sparse_scores - sparse_scores.min()
) / (
    sparse_scores.max() - sparse_scores.min() + 1e-8
)

# Combine Sparse + Dense scores
alpha = 0.5

ensemble_scores = (
    alpha * sparse_scores
    + (1 - alpha) * dense_scores
)

In [ ]:
results = sorted(
    zip(documents, ensemble_scores),
    key=lambda x: x[1],
    reverse=True
)

for i, (doc, score) in enumerate(results, 1):
    print(f"\n--- Rank {i} | Score: {score:.4f} ---")
    print(doc)


--- Rank 1 | Score: 0.9338 ---
Python is widely used for machine learning.

--- Rank 2 | Score: 0.4158 ---
Python can be used to build AI applications.

--- Rank 3 | Score: 0.2931 ---
Pandas is used for data analysis.

--- Rank 4 | Score: 0.2543 ---
Machine learning models learn patterns from data.

--- Rank 5 | Score: 0.2184 ---
SQL is used to manage databases.
